# T-E2：AI Agent 自动生成股市周报 — 数据获取

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI Agent 自动生成股市周报 |
| 小组   | Team02 |
| 成员   | 王佳（25210244）、刘雄（25210196）、黎㵆筠（25210155）、郑嘉豪（25210307）、刘子瑜（25210198）、陈春洁（25210115）、王占溪（25210249）、黎沛鑫（25210156） |
| GitHub | https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report |
| Pages  | https://htmlpreview.github.io/?https://github.com/lizi586586-code/T-E2-AI-Agent-Automated-Stock-Market-Weekly-Report/blob/main/output/weekly_report.html |
| 日期   | 2026-05-18 |


## 任务说明

本步骤通过 AKShare 接口获取 A 股（上证指数、深证成指、沪深300、创业板指）和美股（道琼斯工业、标普500、纳斯达克综合）的日线数据，时间跨度为 2026-05-11 至 2026-05-18，共 5 个交易日。

数据来源：
- A 股数据：`ak.stock_zh_index_daily()` 
- 美股数据：`ak.index_us_stock_sina()`

In [ ]:
import pandas as pd
import numpy as np
import akshare as ak
from datetime import datetime

print("基础库导入完成")

In [ ]:
# ============================================
# A股数据获取
# 参考接口: https://akshare.akfamily.xyz/data/index/index.html#id3
# ============================================

start_date = datetime(2026, 5, 11).date()
end_date = datetime(2026, 5, 18).date()

a_index_codes = {
    "上证指数": "sh000001",
    "深证成指": "sz399001",
    "沪深300": "sh000300",
    "创业板指": "sz399006",
}

a_share_list = []
for name, code in a_index_codes.items():
    try:
        df = ak.stock_zh_index_daily(symbol=code)
        df["指数名称"] = name
        df = df[(df["date"] >= start_date) & (df["date"] <= end_date)]
        a_share_list.append(df)
        print(f"  [{name}] 获取成功，{len(df)} 条记录")
    except Exception as e:
        print(f"  [{name}] 获取失败: {e}")

df_a_share = pd.concat(a_share_list, ignore_index=True) if a_share_list else pd.DataFrame()
print(f"\nA股数据获取完成，共 {len(df_a_share)} 条记录")

### A股数据获取结果

成功获取 4 个指数各 6 条日线记录（含 5 月 18 日当日），字段包括：开盘价、最高价、最低价、收盘价、成交量。数据已按日期筛选，仅保留本周区间。

In [ ]:
# ============================================
# 美股数据获取
# 参考接口: https://akshare.akfamily.xyz/data/index/index.html#id15
# ============================================

us_index_codes = {
    "道琼斯工业": ".DJI",
    "标普500": ".INX",
    "纳斯达克综合": ".IXIC",
}

us_share_list = []
for name, code in us_index_codes.items():
    try:
        df = ak.index_us_stock_sina(symbol=code)
        df["指数名称"] = name
        df = df[(df["date"] >= start_date) & (df["date"] <= end_date)]
        us_share_list.append(df)
        print(f"  [{name}] 获取成功，{len(df)} 条记录")
    except Exception as e:
        print(f"  [{name}] 获取失败: {e}")

df_us_share = pd.concat(us_share_list, ignore_index=True) if us_share_list else pd.DataFrame()
print(f"\n美股数据获取完成，共 {len(df_us_share)} 条记录")

### 美股数据获取结果

成功获取 3 个指数各 5 条日线记录。美股 T+1 结算，最新数据较 A 股少一个交易日。字段结构与 A 股基本一致。

In [ ]:
# ============================================
# 保存原始数据到 data_raw 文件夹
# ============================================

df_a_share.to_csv("data_raw/A股原始数据.csv", index=False, encoding="utf-8-sig")
print("A股原始数据保存完成 → data_raw/A股原始数据.csv")

df_us_share.to_csv("data_raw/美股原始数据.csv", index=False, encoding="utf-8-sig")
print("美股原始数据保存完成 → data_raw/美股原始数据.csv")

### 数据保存

原始数据以 UTF-8 BOM 编码保存为 CSV 文件，存放于 `data_raw/` 目录。A 股和美股分开存储，便于后续清洗和分析时独立处理。

In [ ]:
# ============================================
# 简单查看：行数、列数、前 5 行
# ============================================

print("=" * 50)
print("A股原始数据概览")
print("=" * 50)
print(f"行数: {df_a_share.shape[0]}, 列数: {df_a_share.shape[1]}")
print(f"列名: {list(df_a_share.columns)}")
print("\n前 5 行:")
display(df_a_share.head())

print("\n")
print("=" * 50)
print("美股原始数据概览")
print("=" * 50)
print(f"行数: {df_us_share.shape[0]}, 列数: {df_us_share.shape[1]}")
print(f"列名: {list(df_us_share.columns)}")
print("\n前 5 行:")
display(df_us_share.head())

## 结果解读

本周 A 股整体呈震荡下行走势，上证指数从 4225 点回落至 4131 点；美股三大指数也小幅回调。数据获取流程正常，无异常缺失。数据将移交下一步进行清洗处理。